In [80]:
import numpy as np
import pandas as pd

#TensorFlow packages for building DRNN layer
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import SimpleRNN, Dense, Input, Reshape, LSTM, BatchNormalization
from tensorflow.keras.models import Sequential

#Plotting graph
import matplotlib.pyplot as plt

#Plotting Keras Model
from tensorflow.keras.utils import plot_model

#Normalization of data
from sklearn.preprocessing import StandardScaler

#Utility Module for computing and displaying metrics
from utility_functions import metrics

import warnings
warnings.filterwarnings("ignore")


In [99]:
weather = pd.read_csv('Dataset_weather.csv',parse_dates=[0], index_col=0)

#Extracting dataset into train, validation and test sets
train = weather[:6097]
valid = weather[6097:7404]
test = weather[7404:]


scaler_input_wind = StandardScaler()
scaler_output_wind = StandardScaler()

#Extracting solar and wind X, y columns per set
X_solar_train = train[['SWTDN', 'SWGDN', 'T']]
y_solar_train = train['DE_solar_generation_actual']
X_solar_valid = valid[['SWTDN', 'SWGDN', 'T']]
y_solar_valid = valid['DE_solar_generation_actual']
X_solar_test = test[['SWTDN', 'SWGDN', 'T']]
y_solar_test = test['DE_solar_generation_actual']

X_wind_train = scaler_input_wind.fit_transform(train[['v1', 'v2', 'v_50m', 'z0']])
y_wind_train = scaler_output_wind.fit_transform(train['DE_wind_generation_actual'].values.reshape(-1,1))
X_wind_valid = scaler_input_wind.fit_transform(valid[['v1', 'v2', 'v_50m', 'z0']])
y_wind_valid = scaler_output_wind.fit_transform(valid['DE_wind_generation_actual'].values.reshape(-1,1))
X_wind_test = scaler_input_wind.fit_transform(test[['v1', 'v2', 'v_50m', 'z0']])
y_wind_test = scaler_output_wind.fit_transform(test['DE_wind_generation_actual'].values.reshape(-1,1))

In [100]:
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingRegressor

solarRegressor = HistGradientBoostingRegressor().fit(X_solar_train, y_solar_train)

In [101]:
windRegressor = Sequential()
windRegressor.add(Input(shape=(4,)))
windRegressor.add(Reshape((-1,1)))
windRegressor.add(SimpleRNN(100, activation = 'tanh', input_shape = X_wind_train.shape, return_sequences=False))
windRegressor.add(Dense(100))
windRegressor.add(BatchNormalization())
windRegressor.add(Dense(1))
windRegressor.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape_9 (Reshape)             │ (None, 4, 1)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_9 (SimpleRNN)        │ (None, 100)            │        10,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,801 (81.25 KB)

 Trainable params: 20,601 (80.47 KB)

 Non-trainable params: 200 (800.00 B)

In [102]:
windRegressor.compile(optimizer = 'adam', loss = 'mean_squared_error')
windRegressor.fit(X_wind_train, y_wind_train, validation_data=(X_wind_valid, y_wind_valid), epochs=20, verbose = 0)

In [103]:
total_train_pred = scaler_output_wind.inverse_transform(
    windRegressor.predict(X_wind_train)) + solarRegressor.predict(X_solar_train).reshape((-1,1))
total_val_pred = scaler_output_wind.inverse_transform(
    windRegressor.predict(X_wind_valid)) + solarRegressor.predict(X_solar_valid).reshape((-1,1))
total_test_pred = scaler_output_wind.inverse_transform(
    windRegressor.predict(X_wind_test)) + solarRegressor.predict(X_solar_test).reshape((-1,1))

191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


In [104]:
metrics('drnn', test[['total_renewable_generation']], total_test_pred)

,MAE,MSE,RMSE,RMSLE,R2
model,,,,,
drnn,2660.275333,1.112715e+07,3335.737625,8.112749,0.85295


In [117]:
def predict_wind_energy_scaled(windRegressor, scaler_input_wind, scaler_output_wind, v1, v2, v_50m, z0):
    input_array = np.array([[v1, v2, v_50m, z0]])  
    input_scaled = scaler_input_wind.transform(input_array)   
    input_scaled = input_scaled.reshape((1, 4))  # Shape: (1, 1, 4)
    pred_scaled = windRegressor.predict(input_scaled, verbose=0)   
    pred_actual = scaler_output_wind.inverse_transform(pred_scaled)
    return pred_actual[0][0]


In [118]:
def predict_solar_energy_scaled(solarRegressor, SWTDN, SWGDN, T):
    input_array = np.array([[SWTDN, SWGDN, T]])  
    pred_solar = solarRegressor.predict(input_array)  
    return pred_solar[0]

In [122]:
weather.iloc[33,:12]

DE_wind_generation_actual        17888.0
DE_solar_generation_actual        1341.0
v1                              5.352578
v2                              7.368359
v_50m                           8.990469
h1                              2.542969
h2                             10.542969
z0                              0.163267
SWTDN                         289.254883
SWGDN                          82.102158
T                             273.486023
rho                             1.257853
Name: 2016-01-02 09:00:00+00:00, dtype: object

In [124]:
predict_wind_energy_scaled(windRegressor,scaler_input_wind,
    scaler_output_wind,5.352578,7.368359,8.990469,0.163267)

23194.357

In [126]:
predict_solar_energy_scaled(solarRegressor,289.254883,82.102158,273.486023)


1836.4312315444329